In [ ]:
import pandas as pd

listings = pd.read_csv(
    "../outputs/clean_listings_with_rates.csv",
    low_memory=False
)

sold = pd.read_csv(
    "../outputs/clean_sold_with_rates.csv",
    low_memory=False
)

print("Listings:", listings.shape)
print("Sold:", sold.shape)

## Part 2: Convert Date Columns

To ensure accurate time-based analysis, all date-related fields are converted to the `datetime` data type.

Using a consistent datetime format allows us to:
- Perform date calculations and comparisons.
- Create time-based features.
- Validate the logical order of transactions in later steps.

In [ ]:
date_columns = [
    "CloseDate",
    "PurchaseContractDate",
    "ListingContractDate",
    "ContractStatusChangeDate"
]

print("Listings date columns:")
for col in date_columns:
    if col in listings.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col}")

print("\nSold date columns:")
for col in date_columns:
    if col in sold.columns:
        print(f"✓ {col}")
    else:
        print(f"✗ {col}")

In [ ]:
for col in date_columns:
    if col in listings.columns:
        listings[col] = pd.to_datetime(
            listings[col],
            errors="coerce"
        )

    if col in sold.columns:
        sold[col] = pd.to_datetime(
            sold[col],
            errors="coerce"
        )

In [ ]:
print("Listings date types:")
print(listings[date_columns].dtypes)

print("\nSold date types:")
print(sold[date_columns].dtypes)

## Part 3: Date Consistency Checks

After converting all date fields to the datetime format, we validate whether the transaction timeline follows the expected business process.

The expected order is:

ListingContractDate → PurchaseContractDate → CloseDate

Three boolean flags are created to identify records that violate this timeline.

In [ ]:
print("Listings:")
print(listings.columns.tolist())

print("\nSold:")
print(sold.columns.tolist())

In [ ]:
sold["listing_after_close_flag"] = (
    sold["ListingContractDate"] > sold["CloseDate"]
)

In [ ]:
sold["purchase_after_close_flag"] = (
    sold["PurchaseContractDate"] > sold["CloseDate"]
)

In [ ]:
sold["negative_timeline_flag"] = (
    sold["PurchaseContractDate"] < sold["ListingContractDate"]
)

In [ ]:
print("Listing after Close:")
print(sold["listing_after_close_flag"].sum())

print()

print("Purchase after Close:")
print(sold["purchase_after_close_flag"].sum())

print()

print("Negative Timeline:")
print(sold["negative_timeline_flag"].sum())

### Results

The timeline validation identified a small number of records with inconsistent transaction dates.

- **68 records** have a listing contract date later than the closing date.
- **252 records** have a purchase contract date later than the closing date.
- **288 records** have a purchase contract date earlier than the listing contract date.

These records were flagged for further review rather than removed because they may represent data entry errors, delayed updates, or special transaction cases.

In [ ]:
listing_after_close = sold[
    sold["listing_after_close_flag"]
]

purchase_after_close = sold[
    sold["purchase_after_close_flag"]
]

negative_timeline = sold[
    sold["negative_timeline_flag"]
]

print(listing_after_close.head())
print(purchase_after_close.head())
print(negative_timeline.head())

In [ ]:
sold["date_issue_flag"] = (
    sold["listing_after_close_flag"] |
    sold["purchase_after_close_flag"] |
    sold["negative_timeline_flag"]
)

print("Total records with any date issue:",
      sold["date_issue_flag"].sum())

In [ ]:
sold.to_csv(
    "../outputs/final_clean_sold.csv",
    index=False
)

listings.to_csv(
    "../outputs/final_clean_listings.csv",
    index=False
)